# The reserved period: checks, then one scoring

[Open in Colab](https://colab.research.google.com/github/Gaire-commits/entropy-gate-pipeline/blob/main/notebooks/reserved_evaluation.ipynb)

Two frozen protocols share the period 2024-01-03 to 2026-10-12: `protocols/volatility_v2.yaml` and `protocols/portfolio_v2.yaml`. **It is reserved for these specifications but previously exposed through related research**; `protocols/reserved_period_disclosure.md` lists every prior use.

**This notebook is a record: nothing in it runs any more.** Each section shows when it ran and the code it ran; the results are in `RESULTS.md`. The forward test, the one thing still running, has its own notebook: [forward_test.ipynb](https://colab.research.google.com/github/Gaire-commits/entropy-gate-pipeline/blob/main/notebooks/forward_test.ipynb).

## Step 1: the data checks

*Ran 2026-11-03; after amendment 2, all four checks passed. Results in `RESULTS.md`. The code it ran, kept for the record:*

```python
if STEP >= 1:
    !python scripts/fetch_reference.py --universe universe/sp500.csv --start 2016-01-04 --end 2026-10-13 --out {REF} --parts daily,corporate_actions,meta,auctions 2>&1 | tail -4
    !python scripts/official_prices.py --reference {REF}
    !python scripts/protocol_checks.py --reference {REF}
else:
    print('STEP is 0 - the large-cap checks were skipped')
```

## Step 2: the baseline gate on the official open

*Ran 2026-11-03; the gate passed on the official open. Results in `RESULTS.md`. The code it ran, kept for the record:*

```python
if STEP >= 2:
    # daily bars only (protocols/amendment_4.md): no Drive, no 5-minute cache
    !python scripts/vol_forecast.py --stage baseline --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('STEP is', STEP, '- skipped')
```

## Step 3: scoring the reserved period, once

*Ran 2026-11-17: the portfolio protocol, then the volatility protocol from daily bars. Both showed no improvement from PE. Results in `RESULTS.md`. The code it ran, kept for the record:*

```python
if STEP >= 3:
    assert CONFIRM == 'score the reserved period', "set CONFIRM exactly to run this step"
    # The portfolio protocol was scored on 2026-11-17 and is not run again. Volatility on daily bars only (protocols/amendment_4.md).
    print('=' * 30, 'VOLATILITY')
    !python scripts/vol_forecast.py --stage reserved --score-reserved --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('STEP is', STEP, '- the reserved period was not scored')
```

## Exploration (not registered tests)

Both protocols are scored, so the data from 2016 to 2026 is open for exploring. Anything found is a lead: it is written as a rule and tested on trading days after 2026-11-17. Every test run is counted. Daily bars only, no Drive.

- `'signals'`: ten simple daily features against the next day's return, the next day's open-to-close return, the next two days' return, and next-day variance beyond what volatility history predicts. About 40 tests, with a false-discovery correction.
- `'pe_long'`: PE once the long-run volatility level is in the model (run 2026-11-17; recorded in RESULTS.md).
- `'har_x'`: HAR plus the long-run level, then adding the signal search's features in blocks (the range against its usual level, recent returns, the largest return, volume shock), each measured out of sample, 2017 to 2026.

*Ran 2026-11-17 ('pe_long', 'signals') and 2026-11-18 ('har_x'). Results in `RESULTS.md`. The code it ran, kept for the record:*

```python
if EXPLORE == 'har_x':
    !python scripts/explore_har_x.py --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
elif EXPLORE == 'signals':
    !python scripts/explore_signals.py --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
elif EXPLORE in ('pe_long', True):
    !python scripts/explore_pe_long.py --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('EXPLORE is off')
```

## Replication: the volatility model on mid caps (frozen 2026-11-18)

The model found on large caps (HAR plus the long-run level, recent returns and volume) is frozen in `protocols/har_x_replication_v1.yaml` and run once on the S&P MidCap 400, whose daily volatility has not been used for anything. It downloads the mid caps' daily bars and auction prints, takes the official open, and judges the model by the protocol's rule. Daily bars only, no Drive. Paste the output into the chat.

*Ran 2026-11-18. VERDICT: REPLICATES. Results in `RESULTS.md`. The code it ran, kept for the record:*

```python
if MIDCAP:
    MID = '/content/midcap/reference'
    !python scripts/fetch_reference.py --universe universe/sp400.csv --start 2016-01-04 --end 2026-10-13 --out {MID} --parts daily,corporate_actions,meta,auctions 2>&1 | tail -4
    !python scripts/official_prices.py --reference {MID} 2>&1 | grep -A14 "coverage by year"
    !python scripts/har_x_replication.py --reference {MID} --official {MID}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('MIDCAP is off')
```